## Обучение бустинговых моделей

In [ ]:
!pip install catboost
from catboost import CatBoostClassifier, CatBoostRegressor, Pool
!pip install lightgbm
from lightgbm import LGBMRegressor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 5.7 MB/s eta 0:00:00


### LightGBM

In [ ]:
# Обучение
start_time = time.time()

lgb_model = LGBMRegressor(
    random_state=42,
    n_jobs=-1,  # использовать все ядра
    verbose=1   # чтобы видеть процесс
)

lgb_model.fit(X_train, y_train)

end_time = time.time()

# Предсказания и метрика
lgb_preds = lgb_model.predict(X_test)
lgb_r2 = r2_score(y_test, lgb_preds)



print(f"[LightGBM baseline] Обучение завершено за {end_time - start_time:.2f} секунд.")
print(f"[LightGBM baseline] R² на тестовой выборке: {lgb_r2:.5f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.567828 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1572
[LightGBM] [Info] Number of data points in the train set: 1448774, number of used features: 13
[LightGBM] [Info] Start training from score 0.481988
[LightGBM baseline] Обучение завершено за 31.75 секунд.
[LightGBM baseline] R² на тестовой выборке: 0.95196


In [ ]:
#Сохранение
joblib.dump(lgb_model, f'{save_dir}/lightgbm_model.pkl')

In [ ]:
# Предсказания на тестовой выборке
lgb_preds = lgb_model.predict(X_test)

# Вычисление MAPE
lgb_mape = mean_absolute_percentage_error(y_test, lgb_preds) * 100  # в процентах

print(f"MAPE для LightGBM после удаления нулей: {lgb_mape:.2f}%")


MAPE для LightGBM после удаления нулей: 22.99%


### CatBoost

In [ ]:
cat_features = [
    'researchDayType',
    'programAgeRestrictionName',
    'breaksPrimeTimeStatusName',
]

In [ ]:
cat_model = CatBoostRegressor(
    verbose=100,            # Печать прогресса
    random_state=42,        # Для воспроизводимости
    cat_features=cat_features  # Категориальные признаки
)

# Обучение модели
start_time = time.time()
cat_model.fit(X_train, y_train, eval_set=(X_test, y_test))
end_time = time.time()

# Предсказания и R²
cat_preds = cat_model.predict(X_test)
cat_r2 = r2_score(y_test, cat_preds)

print(f"[CatBoost] Обучение завершено за {end_time - start_time:.2f} секунд.")
print(f"[CatBoost] R² на тестовой выборке: {cat_r2:.5f}")

Learning rate set to 0.159651
0:	learn: 0.3091942	test: 0.3042550	best: 0.3042550 (0)	total: 1.36s	remaining: 22m 41s
100:	learn: 0.0817287	test: 0.0819580	best: 0.0819580 (100)	total: 1m 16s	remaining: 11m 25s
200:	learn: 0.0770960	test: 0.0778221	best: 0.0778221 (200)	total: 2m 31s	remaining: 10m 3s
300:	learn: 0.0746957	test: 0.0757386	best: 0.0757386 (300)	total: 3m 45s	remaining: 8m 42s
400:	learn: 0.0732050	test: 0.0742234	best: 0.0742219 (399)	total: 4m 58s	remaining: 7m 26s
500:	learn: 0.0721577	test: 0.0732110	best: 0.0732110 (500)	total: 6m 13s	remaining: 6m 11s
600:	learn: 0.0713770	test: 0.0725767	best: 0.0725767 (600)	total: 7m 26s	remaining: 4m 56s
700:	learn: 0.0707356	test: 0.0720662	best: 0.0720662 (700)	total: 8m 37s	remaining: 3m 40s
800:	learn: 0.0701900	test: 0.0716004	best: 0.0715996 (799)	total: 9m 51s	remaining: 2m 26s
900:	learn: 0.0696643	test: 0.0711113	best: 0.0711113 (900)	total: 11m 5s	remaining: 1m 13s
999:	learn: 0.0692264	test: 0.0707907	best: 0.0707907

In [ ]:
# Сохраняем модель
cat_model_path = f'{save_dir}/catboost_model.cbm'
cat_model.save_model(cat_model_path)
print(f"[CatBoost] Модель сохранена в: {cat_model_path}")

[CatBoost] Модель сохранена в: /content/drive/MyDrive/Colab Notebooks/TVR_prediction/TVR_models/catboost_model.cbm


In [ ]:
cat_preds = cat_model.predict(X_test)

cat_mape = mean_absolute_percentage_error(y_test, cat_preds) * 100  # в процентах
print(f"MAPE для CatBoost после удаления нулей: {cat_mape:.2f}%")

MAPE для CatBoost после удаления нулей: 21.21%
